In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, lfilter
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# OPTIMAL INVERSE FILTER DESIGN
# LEAST-SQUARES APPROXIMATION
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.oi-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.oi-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.oi-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.oi-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.oi-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:5px;
}

.oi-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.oi-col{
    flex:1;
    min-width:0;
}

.oi-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.oi-code{
    font-family:Consolas,monospace;
    font-size:13px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="oi-root">

<div class="oi-header">
Optimal Inverse Filter Design by Least Squares
</div>

<div class="oi-doc">

The basic idea of the optimal inverse-filter method is different from direct
impulse-response matching.

The desired system <b>H<sub>d</sub>(z)</b> is followed by an approximate inverse
system <b>1/H(z)</b>. If the inverse were exact, the cascade would become the
identity system and its response to an impulse would be exactly

<div style="text-align:center;font-size:15.5px;margin:6px 0;">
<b>y[n] = δ[n].</b>
</div>

For an all-pole approximation

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
H(z) = β₀ /
(1 + α₁z<sup>-1</sup> + ... + α<sub>N</sub>z<sup>-N</sup>),
</b>
</div>

the inverse-filter output is

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
y[n] =
(1/β₀)
[h<sub>d</sub>[n] +
Σ αₖh<sub>d</sub>[n-k]].
</b>
</div>

The denominator coefficients αₖ are selected so that the energy of the
non-impulsive part of this output is minimized. This leads to a linear
least-squares system involving correlation quantities ξ(k,l).

The example below approximates the same ideal low-pass response used in the
previous Padé example, using an all-pole model with <b>N = 5</b>.

</div>

</div>
"""))

# ============================================================
# DESIRED IMPULSE-RESPONSE SAMPLES
# ============================================================

hd = np.array([0.063661,0.000000,-0.106103,0.000000,0.318309,0.500000,0.318309,0.000000,-0.106103,0.000000,0.063661])

N = 5

beta0 = hd[0]

# ============================================================
# CORRELATION SYSTEM OF THE NUMERICAL EXAMPLE
# ============================================================

Xi = np.array([
    [0.483262,0.318309,0.027019,-0.053052,-0.013510],
    [0.318309,0.483262,0.318309,0.027019,-0.053052],
    [0.027019,0.318309,0.483262,0.318309,0.027019],
    [-0.053052,0.027019,0.318309,0.483262,0.318309],
    [-0.013510,-0.053052,0.027019,0.318309,0.483262]
])

gamma = np.array([-0.318309,-0.027019,0.053052,0.013510,-0.031830])

# ============================================================
# SOLVE FOR DENOMINATOR COEFFICIENTS
# ============================================================

alpha = np.linalg.solve(Xi,gamma)

a = np.concatenate(([1.0],alpha))

b = np.array([beta0])

# ============================================================
# POLES
# ============================================================

poles = np.roots(a)

pole_radius = np.abs(poles)

max_pole_radius = np.max(pole_radius)

stable = max_pole_radius < 1.0

# ============================================================
# APPROXIMATE INVERSE OUTPUT FOR THE AVAILABLE SAMPLES
# ============================================================

L = len(hd)

n = np.arange(L)

y_inverse = np.zeros(L)

for sample_index in range(L):

    value = hd[sample_index]

    for k in range(1,N+1):

        if sample_index-k >= 0:

            value += alpha[k-1]*hd[sample_index-k]

    y_inverse[sample_index] = value/beta0

delta = np.zeros(L)

delta[0] = 1.0

inverse_error = y_inverse-delta

E_inverse = np.sum(inverse_error[1:]**2)

# ============================================================
# IMPULSE RESPONSE OF THE ALL-POLE MODEL
# ============================================================

Lh = 18

impulse = np.zeros(Lh)

impulse[0] = 1.0

h_model = lfilter(b,a,impulse)

# ============================================================
# DESIRED LOW-PASS IMPULSE RESPONSE
# ============================================================

nh = np.arange(Lh)

hd_long = np.zeros(Lh)

for i,sample_index in enumerate(nh):

    if sample_index == 5:

        hd_long[i] = 0.5

    else:

        hd_long[i] = np.sin((sample_index-5)*np.pi/2)/(np.pi*(sample_index-5))

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

omega,H = freqz(b,a,worN=32768)

omega_norm = omega/np.pi

mag = np.abs(H)

ideal_mag = np.where(omega <= np.pi/2,1.0,0.0)

# ============================================================
# DISPLAY — NUMERICAL EXAMPLE
# ============================================================

display(HTML(f"""
<div class="oi-root">

<div class="oi-box oi-note">

<div class="oi-title">Numerical example — Ideal low-pass approximation</div>

The first eleven desired impulse-response samples are

<div class="oi-code" style="margin-top:5px;text-align:center;">
[0.063661, 0, -0.106103, 0, 0.318309, 0.500000,
0.318309, 0, -0.106103, 0, 0.063661]
</div>

The approximate model contains <b>N = 5 poles</b> and no finite zeros.

The scale factor is

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>β₀ = h<sub>d</sub>[0] = {beta0:.6f}.</b>
</div>

</div>

<div class="oi-box">

<div class="oi-title">Calculated all-pole model</div>

<div class="oi-cols">

<div class="oi-col">

<b>Denominator coefficients</b><br>

α₀ = {a[0]:.6f}<br>
α₁ = {a[1]:.6f}<br>
α₂ = {a[2]:.6f}<br>
α₃ = {a[3]:.6f}<br>
α₄ = {a[4]:.6f}<br>
α₅ = {a[5]:.6f}

</div>

<div class="oi-col">

<b>Model information</b><br>

β₀ = <b>{beta0:.6f}</b><br><br>

Maximum pole radius:<br>

<b>{max_pole_radius:.6f}</b><br><br>

Stable:
<b>{"YES" if stable else "NO"}</b>

</div>

<div class="oi-col">

<b>Inverse-system criterion</b><br>

Desired cascade output:<br>

<b>δ[n]</b><br><br>

Computed error energy:<br>

<b>E = {E_inverse:.6f}</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# TRANSFER FUNCTION
# ============================================================

display(HTML(f"""
<div class="oi-root">

<div class="oi-box oi-note">

<div class="oi-title">Resulting transfer function</div>

<div style="font-size:14.5px;line-height:1.65;white-space:nowrap;">

H(z) =
<b>
{beta0:.6f} /
(1
{a[1]:+.6f}z<sup>-1</sup>
{a[2]:+.6f}z<sup>-2</sup>
{a[3]:+.6f}z<sup>-3</sup>
{a[4]:+.6f}z<sup>-4</sup>
{a[5]:+.6f}z<sup>-5</sup>)
</b>

</div>

<div style="margin-top:6px;">

Because the approximation contains only poles, its ability to reproduce the
ideal low-pass response is limited. The selected order is also only one
particular trial choice.

</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.4))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. DESIRED CASCADE OUTPUT VS ACTUAL OUTPUT
# ============================================================

marker_delta,stem_delta,base_delta = ax1.stem(n,delta,linefmt='C0-',markerfmt='C0o',basefmt=' ')

marker_y,stem_y,base_y = ax1.stem(n,y_inverse,linefmt='r--',markerfmt='ro',basefmt=' ')

plt.setp(stem_delta,linewidth=1.1)

plt.setp(stem_y,linewidth=1.0)

marker_delta.set_markersize(4.5)

marker_y.set_markersize(3.5)

marker_delta.set_label(r'Desired $\delta[n]$')

marker_y.set_label(r'Actual $y[n]$')

ax1.axhline(0,color='black',linewidth=0.8)

ax1.set_xlim(-0.5,L-0.5)

ax1.set_ylim(-4.0,6.0)

ax1.set_title('Inverse-System Output')

ax1.set_xlabel('Sample index n')

ax1.set_ylabel('Amplitude')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 2. INVERSE APPROXIMATION ERROR
# ============================================================

marker_e,stem_e,base_e = ax2.stem(n,inverse_error,linefmt='r-',markerfmt='ro',basefmt=' ')

plt.setp(stem_e,linewidth=1.1)

marker_e.set_markersize(4.5)

ax2.axhline(0,color='black',linewidth=0.8)

ax2.set_xlim(-0.5,L-0.5)

ax2.set_ylim(-4.0,6.0)

ax2.set_title(r'Inverse Error $y[n]-\delta[n]$')

ax2.set_xlabel('Sample index n')

ax2.set_ylabel('Error')

ax2.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# 3. DESIRED VS ALL-POLE IMPULSE RESPONSE
# ============================================================

ax3.plot(nh,hd_long,color='black',linewidth=1.2,label=r'Desired $h_d[n]$')

ax3.plot(nh,h_model,color='red',linewidth=1.3,label=r'All-pole $h[n]$')

ax3.axhline(0,color='black',linewidth=0.8)

ax3.set_xlim(0,Lh-1)

ax3.set_ylim(-150,300)

ax3.set_title('Desired vs All-Pole Impulse Response')

ax3.set_xlabel('Sample index n')

ax3.set_ylabel('Amplitude')

ax3.grid(True,linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 4. MAGNITUDE RESPONSE
# ============================================================

ax4.plot(omega_norm,ideal_mag,color='black',linewidth=1.2,label='Ideal low-pass')

ax4.plot(omega_norm,mag,color='red',linewidth=1.4,label='All-pole approximation')

ax4.axvline(0.5,linestyle='--',linewidth=1.0,label=r'$\omega_c=\pi/2$')

ax4.set_xlim(0,1)

ax4.set_ylim(0,1.1)

ax4.set_title('Ideal vs All-Pole Magnitude Response')

ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax4.set_ylabel(r'$|H(e^{j\omega})|$')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.56)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)